In [7]:
import nltk
nltk.download('punkt',quiet=True)
nltk.download("stopwords",quiet=True)
import string 
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import pickle

class Query:
    input_li = []
    op_li = []

    def tokenize_seq(self,new_s):
        new_s = new_s.lower()
        translate_table = dict((ord(char), " ") for char in string.punctuation)   
        new_s = new_s.translate(translate_table)
        li = word_tokenize(new_s)
        stop_words = set(stopwords.words("english"))
        filter_li = []
        for words in li:
            if(words not in stop_words):
                filter_li.append(words)
        return filter_li

    def __init__(self,input_seq,op_seq):
        input_li = self.tokenize_seq(input_seq)
        op_li = []
        first_op_li = []
        if(op_seq != ""):
            first_op_li = op_seq.split(",")
            for i in range(len(first_op_li)):
                temp_l = first_op_li[i].split()
                for j in temp_l:
                    op_li.append(j)
        for i in range(len(op_li)):
            op_li[i] = (op_li[i].strip()).lower()
        self.input_li = input_li
        self.op_li = op_li
    
    def getQuery(self):
        return self.input_li,self.op_li

class Inverted_Index:
    inverted_ind = {}
    universal_set = set()
    comparisons = 0
    document_count = 0
    name_arr = []

    def __init__(self):
        self.inverted_ind = {}
        self.universal_set = set()
        self.comparisons = 0
        self.document_count = 0
        self.name_arr = []
       
    def extract_text(self,s):
        start = [0,0] #Start is inclusive 
        end = [0,0] #End is exclusive
        for i in range(len(s)):
            if(i + 7 <= len(s) and s[i:i+7] == "<TITLE>"):
                start[0] = i+7
            elif(i + 8 <= len(s) and s[i:i+8] == "</TITLE>"):
                end[0] = i
            elif(i + 6 <= len(s) and s[i:i+6] == "<TEXT>"):
                start[1] = i+6
            elif(i + 7 <= len(s) and s[i:i+7] == "</TEXT>"):
                end[1] = i
        new_s = s[start[0]:end[0]] + " " + s[start[1]:end[1]]
        new_s = " ".join(new_s.split("\n"))
        new_s = " ".join(new_s.split("-"))
        return new_s
    
    def addDoc(self,token_list,id): #token list in order, id of parent document 
        for i in range(len(token_list)):
            key = token_list[i]
            if(key in self.inverted_ind.keys()):
                self.inverted_ind[str(key)].append(int(id))
            else:
                li = [int(id)]
                self.inverted_ind[str(key)] = li
            self.universal_set.add(int(id))
                
    def new_Data(self,path):
        f = open(path,"r")
        s = f.read()
        new_s = self.extract_text(s)
        new_s = new_s.lower()
        translate_table = dict((ord(char), " ") for char in string.punctuation)   
        new_s = new_s.translate(translate_table)
        f.close()
        li = word_tokenize(new_s)
        stop_words = set(stopwords.words("english"))
        filter_li = []
        for words in li:
            if(words not in stop_words):
                filter_li.append(words)
        self.addDoc(filter_li,self.document_count)
        self.name_arr.append(path)
        self.document_count+=1

    def showWord(self,key):
        return self.inverted_ind[str(key)]
    
    def getFreq(self,key):
        return len(self.inverted_ind[key])

    def simplify_not(self,op_seq):
        if(op_seq == []):
            return op_seq
        simplified_seq = []
        not_count = 1
        for i in range(len(op_seq)-1):
            if(op_seq[i+1] == op_seq[i] and op_seq[i] == "not"):
                not_count+=1
            else:
                if(op_seq[i] == "not"):
                    if(not_count%2 == 1):
                        simplified_seq.append(op_seq[i])
                    not_count = 1
                else:
                    simplified_seq.append(op_seq[i])
        if(op_seq[len(op_seq)-1] == "not"):
            if(not_count%2 == 1):
                simplified_seq.append(op_seq[len(op_seq)-1])
        else:
            simplified_seq.append(op_seq[len(op_seq)-1])
        return simplified_seq
    
    def processQuery(self,input_seq,op_seq):
        self.comparisons = 0
        query = Query(input_seq,op_seq)
        input_li,op_li = query.getQuery()
        og_qry = self.getStringQuery(input_li,op_li)
        op_li = self.simplify_not(op_li)
        str_qry = self.getStringQuery(input_li,op_li)
        for i in range(len(input_li)):
            if(input_li[i] in self.inverted_ind.keys()):
                input_li[i] = self.inverted_ind[input_li[i]]
            else:
                input_li[i] = []
        output = self.query_sched(input_li,op_li)
        comp_ans = self.comparisons
        self.comparisons = 0
        return output,comp_ans,str_qry,og_qry
    
    def getStringQuery(self,input_li,op_li):
        ans = []
        i = 0
        j = 0
        while(i < len(op_li) and j < len(input_li)):
            if(op_li[i] == "not"):
                ans.append(op_li[i])
                i+=1
            else:
                ans.append(input_li[j])
                j+=1
                ans.append(op_li[i])
                i+=1
        if(j < len(input_li)):
            ans.append(input_li[j])
        final_ans = " ".join(ans)
        return final_ans

    def getOutput(self,input_seq,op_seq):
        output,comp_ans,str_qry,og_qry = self.processQuery(input_seq,op_seq)
        print("Original Input Query:",og_qry)
        print("Simplified Input Query:",str_qry)
        print("Number of Documents:",len(output[0]))
        if(len(output[0])>0):
            print("Document Names:")
            for j in range(len(output[0])):
                print("\t",str(j+1)+".",self.name_arr[output[0][j]])
        else:
            print("No documents to show!")
        print("Number of comparisons for fetching result:",comp_ans)

    
    def query_sched(self,input_li,op_li): #input_li is list of lists, each list in input_li is the doc_list of a regex
        if(len(op_li) == 0):
            return input_li
        elif(op_li[0] == "not"):
            output = self.query_not(input_li[0])
            return self.query_sched([output]+input_li[1:],op_li[1:])
        elif(op_li[0] == "and"):
            if(len(op_li) > 1 and op_li[1] == "not"):
                op_li[0] = "not"
                op_li[1] = "and"
                temp = input_li[0]
                input_li[0] = input_li[1]
                input_li[1] = temp
                return self.query_sched(input_li,op_li)
            else:
                output = self.query_and(input_li[0],input_li[1])
        elif(op_li[0]  == "or"):
            if(len(op_li) > 1 and op_li[1] == "not"):
                op_li[0] = "not"
                op_li[1] = "or"
                temp = input_li[0]
                input_li[0] = input_li[1]
                input_li[1] = temp
                return self.query_sched(input_li,op_li)
            else:
                output = self.query_or(input_li[0],input_li[1])
        new_l = [output]+ input_li[2:]
        return self.query_sched(new_l,op_li[1:])

    def query_and(self,t1,t2):
        t1_li = t1
        t2_li = t2
        merge_li = []
        st = 0
        st2 = 0
        while(st<len(t1_li) and st2 < len(t2_li)):
            if(t1_li[st] == t2_li[st2]):
                merge_li.append(t1_li[st])
                st +=1
                st2 +=1
            elif(t1_li[st]<t2_li[st2]):
                st +=1
            else:
                st2+=1
            self.comparisons+=1
        return merge_li

    def query_or(self,a,b):
        a_list = a
        b_list = b
        ait = 0
        bit = 0
        output = []
        while(ait < len(a_list) and bit < len(b_list)):
            if(a[ait] < b[bit]):
                output.append(a[ait])
                ait+=1
            elif(a[ait] == b[bit]):
                output.append(a[ait])
                ait+=1
                bit+=1
            else:
                output.append(b[bit])
                bit+=1
            self.comparisons+=1
        while(ait < len(a_list)):
            output.append(a[ait])
            ait+=1
        while(bit < len(b_list)):
            output.append(b[bit])
            bit+=1
        return output 
    
    def query_not(self,a):
        univ = list(self.universal_set)
        univ.sort()
        output = []
        self.comparisons = 0
        ait = 0
        for i in range(len(univ)):
            if(ait < len(a)):
                self.comparisons+=1
            if(ait < len(a) and univ[i] == a[ait]):
                ait+=1
            else:
                output.append(univ[i])
        return output

def getNum(i):
    return (4-len(str(i)))*"0" + str(i)

try:
    inverted_index = pickle.load(open("unigram_inverted_index_savefile.pickle", "rb"))
except (OSError, IOError) as e:
    inverted_index = Inverted_Index()
    for j in range(1,1401): # 1,1401
        f = inverted_index.new_Data("CSE508_Winter2023_Dataset/cranfield"+getNum(j))
    pickle.dump(inverted_index, open("unigram_inverted_index_savefile.pickle", "wb"))

print("\n++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
n = int(input())
for i in range(n):
    inp_seq = input()
    op_seq = input() 
    output = inverted_index.getOutput(inp_seq,op_seq)
    print("++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")



++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Original Input Query: experimental and flow or shear
Simplified Input Query: experimental and flow or shear
Number of Documents: 366
Document Names:
	 1. CSE508_Winter2023_Dataset/cranfield0001
	 2. CSE508_Winter2023_Dataset/cranfield0002
	 3. CSE508_Winter2023_Dataset/cranfield0002
	 4. CSE508_Winter2023_Dataset/cranfield0003
	 5. CSE508_Winter2023_Dataset/cranfield0004
	 6. CSE508_Winter2023_Dataset/cranfield0004
	 7. CSE508_Winter2023_Dataset/cranfield0009
	 8. CSE508_Winter2023_Dataset/cranfield0016
	 9. CSE508_Winter2023_Dataset/cranfield0017
	 10. CSE508_Winter2023_Dataset/cranfield0019
	 11. CSE508_Winter2023_Dataset/cranfield0025
	 12. CSE508_Winter2023_Dataset/cranfield0035
	 13. CSE508_Winter2023_Dataset/cranfield0045
	 14. CSE508_Winter2023_Dataset/cranfield0050
	 15. CSE508_Winter2023_Dataset/cranfield0052
	 16. CSE508_Winter2023_Dataset/cranfield0053
	 17. CSE508_Winter2023_Dataset/cranfield005

BIGRAM INDEX

In [37]:
import nltk
nltk.download('punkt',quiet=True)
nltk.download("stopwords",quiet=True)
import string 
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import pickle

class Query:
    input_li = []
    # op_li = []

    def tokenize_seq(self,new_s):
        new_s = new_s.lower()
        translate_table = dict((ord(char), " ") for char in string.punctuation)   
        new_s = new_s.translate(translate_table)
        li = word_tokenize(new_s)
        stop_words = set(stopwords.words("english"))
        filter_li = []
        for words in li:
            if(words not in stop_words):
                filter_li.append(words)
        
        filter_li_new = []
        for i in range(len(filter_li)-1):
            filter_li_new.append(filter_li[i]+" "+filter_li[i+1])
            
        return filter_li_new

    def __init__(self,input_seq):
        input_li = self.tokenize_seq(input_seq)
        self.input_li = input_li
    
    def getQuery(self):
        return self.input_li

class Bigram_Inverted_Index:
    inverted_ind = {}
    universal_set = set()
    document_count = 0
    name_arr = []

    def __init__(self):
        self.inverted_ind = {}
        self.universal_set = set()
        self.document_count = 0
        self.name_arr = []
            
    def addDoc(self,token_list,id):
        for i in range(len(token_list)):
            key = token_list[i]
            if(key in self.inverted_ind.keys()):
                self.inverted_ind[str(key)].append(int(id))
            else:
                li = [int(id)]
                self.inverted_ind[str(key)] = li
            self.universal_set.add(int(id))

    def showWord(self,key):
        return self.inverted_ind[str(key)]
    
    def getFreq(self,key):
        return len(self.inverted_ind[key])

    def new_Data(self,path):
        f = open(path,"r")
        s = f.read()
        new_s = self.extract_text(s)
        new_s = new_s.lower()
        translate_table = dict((ord(char), " ") for char in string.punctuation)   
        new_s = new_s.translate(translate_table)
        f.close()
        li = word_tokenize(new_s)
        stop_words = set(stopwords.words("english"))
        filter_li = []
        for words in li:
            if(words not in stop_words):
                filter_li.append(words)
        new_li = filter_li[::]
        filter_li = []
        for i in range(len(new_li)-1):
            filter_li.append(new_li[i]+" "+new_li[i+1])
        filter_li = list(set(filter_li))
        self.addDoc(filter_li,self.document_count)
        self.name_arr.append(path)
        print(self.name_arr)
        self.document_count+=1
    
    def processQuery(self,input_seq):
        query = Query(input_seq)
        input_li = query.getQuery()
        op_helper = []
        for i in range(len(input_li)-1):
            op_helper.append('and')
        str_qry = self.getStringQuery(input_li,op_helper)
        for i in range(len(input_li)):
            if(input_li[i] in self.inverted_ind.keys()):
                input_li[i] = self.inverted_ind[input_li[i]]
            else:
                input_li[i] = []
        output = self.query_sched(input_li,op_helper)
        return output,str_qry
    
    def getStringQuery(self,input_li,op_li):
        ans = []
        i = 0
        j = 0
        while(i < len(op_li) and j < len(input_li)):
            ans.append(input_li[j])
            j+=1
            ans.append(op_li[i])
            i+=1

        if(j < len(input_li)):
            ans.append(input_li[j])
        final_ans = " ".join(ans)
        return final_ans

    def getOutput(self,input_seq, show_names = False):
        output,str_qry = self.processQuery(input_seq)
        print("Input Query:",str_qry)
        # print(str_qry)
        # print("Number of Documents:",len(output[0]))
        # print("Document IDs:",*output[0])
        # print("Number of comparisons for fetching result:",comp_ans)
        if(show_names):
            fin_out = []
            for i in output[0]:
                fin_out.append(self.name_arr[i])
            return fin_out
        return output[0]

    def query_sched(self,input_li,op_li): #input_li is list of lists, each list in input_li is the doc_list of a regex
        if(len(op_li) == 0):
            return input_li
        elif(op_li[0] == "and"):
            output = self.query_and(input_li[0],input_li[1])

        new_l = [output]+ input_li[2:]
        return self.query_sched(new_l,op_li[1:])

    def query_and(self,t1,t2):
        t1_li = t1
        t2_li = t2
        merge_li = []
        st = 0
        st2 = 0
        while(st<len(t1_li) and st2 < len(t2_li)):
            if(t1_li[st] == t2_li[st2]):
                merge_li.append(t1_li[st])
                st +=1
                st2 +=1
            elif(t1_li[st]<t2_li[st2]):
                st +=1
            else:
                st2+=1
            self.comparisons+=1
        return merge_li

    def extract_text(self,s):
        start = [0,0] #Start is inclusive 
        end = [0,0] #End is exclusive

        for i in range(len(s)):
            if(i + 7 <= len(s) and s[i:i+7] == "<TITLE>"):
                start[0] = i+7
            elif(i + 8 <= len(s) and s[i:i+8] == "</TITLE>"):
                end[0] = i
            elif(i + 6 <= len(s) and s[i:i+6] == "<TEXT>"):
                start[1] = i+6
            elif(i + 7 <= len(s) and s[i:i+7] == "</TEXT>"):
                end[1] = i

        new_s = s[start[0]:end[0]] + " " + s[start[1]:end[1]]
        new_s = " ".join(new_s.split("\n"))
        new_s = " ".join(new_s.split("-"))

        return new_s

def getNum(i):
    return (4-len(str(i)))*"0" + str(i)

try:
    invertedIndex = pickle.load(open("bigram_index_savefile.pickle", "rb"))
except (OSError, IOError) as e:
    invertedIndex = Bigram_Inverted_Index()
    for j in range(1,1401): # 1,1401
        f = invertedIndex.new_Data("CSE508_Winter2023_Dataset/cranfield"+getNum(j))
    pickle.dump(invertedIndex, open("bigram_index_savefile.pickle", "wb"))

print("\n++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
# print(inverted_ind)
n = int(input())
for i in range(n):
    inp_seq = input()
    print("Search Phrase:",inp_seq)
    output = invertedIndex.getOutput(inp_seq, show_names = True)
    # output = invertedIndex.getOutput(inp_seq,show_names = True)
    print("Number of Documents Retrieved:",len(output))
    if(len(output)>0):
        print("The following documents contain your search phrase:")
        for j in range(len(output)):
            print("\t",str(j+1)+".",output[j])
    else:
        print("No matches found!")
    print("++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")


{'due slipstream': [1], 'ratios results': [1], 'effect integrated': [1], 'wing propeller': [1], 'theoretical treatments': [1], 'different angles': [1], 'different free': [1], 'flow theory': [1], 'study wing': [1], 'well potential': [1], 'destalling boundary': [1], 'made specific': [1], 'increment subtracting': [1], 'comparative span': [1], 'boundary layer': [1], 'investigation aerodynamics': [1], 'experimental study': [1], 'experimental investigation': [1], 'problem comparative': [1], 'increase due': [1], 'attack wing': [1], 'aerodynamics wing': [1], 'layer control': [1], 'destalling effects': [1], 'propeller slipstream': [1], 'produced slipstream': [1], 'subtracting destalling': [1], 'slipstream different': [1], 'remaining lift': [1], 'agree well': [1], 'stream slipstream': [1], 'velocity ratios': [1], 'free stream': [1], 'showed substantial': [1], 'different theoretical': [1], 'increment produced': [1], 'determine spanwise': [1], 'slipstream made': [1], 'destalling lift': [1], 'part 

POSITIONAL INDEX

In [3]:
import nltk
nltk.download('punkt',quiet=True)
nltk.download("stopwords",quiet=True)
import string 
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import pickle

class Query:
    input_li = []

    def tokenize_seq(self,new_s):
        new_s = new_s.lower()
        translate_table = dict((ord(char), " ") for char in string.punctuation)   
        new_s = new_s.translate(translate_table)
        li = word_tokenize(new_s)
        stop_words = set(stopwords.words("english"))
        filter_li = []
        for words in li:
            if(words not in stop_words):
                filter_li.append(words)
        return filter_li

    def __init__(self,input_seq):
        input_li = self.tokenize_seq(input_seq)
        self.input_li = input_li
    
    def getQuery(self):
        return self.input_li

class Positional_Inverted_Index:
    inverted_ind = {} # {word:dict} #dict{id:list}
    document_count = 0
    name_arr = []

    def extract_text(self,s):
        start = [0,0] #Start is inclusive 
        end = [0,0] #End is exclusive
        for i in range(len(s)):
            if(i + 7 <= len(s) and s[i:i+7] == "<TITLE>"):
                start[0] = i+7
            elif(i + 8 <= len(s) and s[i:i+8] == "</TITLE>"):
                end[0] = i
            elif(i + 6 <= len(s) and s[i:i+6] == "<TEXT>"):
                start[1] = i+6
            elif(i + 7 <= len(s) and s[i:i+7] == "</TEXT>"):
                end[1] = i

        new_s = s[start[0]:end[0]] + " " + s[start[1]:end[1]]
        new_s = " ".join(new_s.split("\n"))
        new_s = " ".join(new_s.split("-"))
        return new_s
    
    def __init__(self):
        self.inverted_ind = {}
        self.document_count = 0
        self.name_arr = []

    def new_Data(self,path):
        f = open(path,"r")
        s = f.read()
        new_s = self.extract_text(s)
        new_s = new_s.lower()
        translate_table = dict((ord(char), " ") for char in string.punctuation)   
        new_s = new_s.translate(translate_table)
        f.close()
        li = word_tokenize(new_s)
        stop_words = set(stopwords.words("english"))
        filter_li = []
        for words in li:
            if(words not in stop_words):
                filter_li.append(words)
        self.addDoc(filter_li,self.document_count)
        self.name_arr.append(path)
        self.document_count+=1
    
    def addDoc(self,token_list,doc_id): #token list in order, id of parent document 
        for i in range(len(token_list)):
            key = token_list[i]
            if(key in self.inverted_ind.keys()):
                dict = self.inverted_ind[str(key)]
                if(int(doc_id) not in dict.keys()):
                    dict[int(doc_id)] = [i]
                else:
                    dict[int(doc_id)].append(i)
            else:
                dict = {}
                dict[int(doc_id)] = [i]
                self.inverted_ind[str(key)] = dict

    def getPositionalList(self,key):
        return self.inverted_ind[str(key)]
        
    def binary_search(self,arr, x):
        low = 0
        high = len(arr) - 1
        mid = 0
        while low <= high:
            mid = (high + low) // 2
            if arr[mid] < x:
                low = mid + 1
            elif arr[mid] > x:
                high = mid - 1
            else:
                return 1
        return 0
    
    def getFreq(self,key):
        return len(self.inverted_ind[key])
    
    def check_existence(self,word,document_id,position):
        if word in self.inverted_ind.keys():
            if document_id in self.inverted_ind[word].keys():
                return self.binary_search(self.inverted_ind[word][document_id],position)
            else:
                return -1
        return -2

    def processHelper(self,input_li):
        output = []
        override = 1
        if input_li[0] not in self.inverted_ind.keys():
            return output
        for cur_doc in self.inverted_ind[input_li[0]].keys():
            for cur_pos in self.inverted_ind[input_li[0]][cur_doc]:
                next_pos = cur_pos+1
                j = 1
                while(j< len(input_li)):
                    x = self.check_existence(input_li[j],cur_doc,next_pos)
                    j+=1
                    next_pos+=1
                    override = x
                    if(override <= 0):
                        break
                if(override <= -1):
                    break
                if(j == len(input_li)):
                    output.append(cur_doc)
            if(override <= -2):
                break
        return output
    
    def getOutput(self,input_seq,show_names = False):
        query = Query(input_seq)
        input_li= query.getQuery()
        output = list(set(self.processHelper(input_li)))
        output.sort()
        if(show_names):
            out_list = []
            for i in output:
                out_list.append(self.name_arr[i])
            return out_list
        return output

def getNum(i):
    return (4-len(str(i)))*"0" + str(i)

try:
    invertedIndex = pickle.load(open("inverted_index_savefile.pickle", "rb"))
except (OSError, IOError) as e:
    invertedIndex = Positional_Inverted_Index()
    for j in range(1,1401): # 1,1401
        f = invertedIndex.new_Data("CSE508_Winter2023_Dataset/cranfield"+getNum(j))
    pickle.dump(invertedIndex, open("inverted_index_savefile.pickle", "wb"))

print("\n++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
n = int(input())
for i in range(n):
    inp_seq = input()
    print("Search Phrase:",inp_seq)
    output = invertedIndex.getOutput(inp_seq,show_names = True)
    print("Number of Documents Retrieved:",len(output))
    if(len(output)>0):
        print("The following documents contain your search phrase:")
        for j in range(len(output)):
            print("\t",str(j+1)+".",output[j])
    else:
        print("No matches found!")
    print("++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")



++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Search Phrase: laminar flow
Number of Documents Retrieved: 173
The following documents contain your search phrase:
	 1. CSE508_Winter2023_Dataset/cranfield0004
	 2. CSE508_Winter2023_Dataset/cranfield0007
	 3. CSE508_Winter2023_Dataset/cranfield0009
	 4. CSE508_Winter2023_Dataset/cranfield0016
	 5. CSE508_Winter2023_Dataset/cranfield0021
	 6. CSE508_Winter2023_Dataset/cranfield0023
	 7. CSE508_Winter2023_Dataset/cranfield0049
	 8. CSE508_Winter2023_Dataset/cranfield0050
	 9. CSE508_Winter2023_Dataset/cranfield0054
	 10. CSE508_Winter2023_Dataset/cranfield0055
	 11. CSE508_Winter2023_Dataset/cranfield0059
	 12. CSE508_Winter2023_Dataset/cranfield0072
	 13. CSE508_Winter2023_Dataset/cranfield0073
	 14. CSE508_Winter2023_Dataset/cranfield0081
	 15. CSE508_Winter2023_Dataset/cranfield0084
	 16. CSE508_Winter2023_Dataset/cranfield0088
	 17. CSE508_Winter2023_Dataset/cranfield0089
	 18. CSE508_Winter2023_Da


++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Search Phrase: two dimensional
Input Query: two dimensional
Number of Documents Retrieved: 162
The following documents contain your search phrase:
	 1. CSE508_Winter2023_Dataset/cranfield0002
	 2. CSE508_Winter2023_Dataset/cranfield0004
	 3. CSE508_Winter2023_Dataset/cranfield0008
	 4. CSE508_Winter2023_Dataset/cranfield0009
	 5. CSE508_Winter2023_Dataset/cranfield0015
	 6. CSE508_Winter2023_Dataset/cranfield0018
	 7. CSE508_Winter2023_Dataset/cranfield0020
	 8. CSE508_Winter2023_Dataset/cranfield0025
	 9. CSE508_Winter2023_Dataset/cranfield0049
	 10. CSE508_Winter2023_Dataset/cranfield0052
	 11. CSE508_Winter2023_Dataset/cranfield0059
	 12. CSE508_Winter2023_Dataset/cranfield0060
	 13. CSE508_Winter2023_Dataset/cranfield0073
	 14. CSE508_Winter2023_Dataset/cranfield0089
	 15. CSE508_Winter2023_Dataset/cranfield0094
	 16. CSE508_Winter2023_Dataset/cranfield0118
	 17. CSE508_Winter2023_Dataset/cranfiel